In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import clone

In [3]:
train = pd.read_csv("../MLOpsedian/data/processed/train_regression_onehot_ready.csv")
test = pd.read_csv("../MLOpsedian/data/processed/test_regression_onehot_ready.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (19700, 1639)
Test shape: (5300, 1638)


Basic SETUP


In [4]:
id_col = "record_id"
target = "flood_risk_score"

X = train.drop(columns=[id_col, target])
y = train[target]

X_test = test.drop(columns=[id_col])
test_ids = test[id_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

print("Columns match:", list(X.columns) == list(X_test.columns))

X shape: (19700, 1637)
y shape: (19700,)
X_test shape: (5300, 1637)
Columns match: True


In [5]:
os.makedirs("../submissions", exist_ok=True)

In [6]:
def validate_submission(submission):
    print("Submission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

Mean Baseline Submission

In [7]:
train_mean = y.mean()

mean_preds = np.full(len(test_ids), train_mean)

sub_mean = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": mean_preds
})

validate_submission(sub_mean)

sub_mean.to_csv("../submissions/sub_v001_mean_baseline.csv", index=False)

print("Saved: ../submissions/sub_v001_mean_baseline.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.478517
1   F100765          0.478517
2   F107573          0.478517
3   F110345          0.478517
4   F118850          0.478517

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.47851742385786805
Max: 0.47851742385786805

Submission is valid.
Saved: ../submissions/sub_v001_mean_baseline.csv


Cross Validation

In [8]:
def cross_validate_model(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    rmse_scores = []
    mae_scores = []
    r2_scores = []
    
    for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        
        fold_model = clone(model)
        fold_model.fit(X_train, y_train)
        
        valid_preds = fold_model.predict(X_valid)
        valid_preds = np.clip(valid_preds, 0, 1)
        
        rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
        mae = mean_absolute_error(y_valid, valid_preds)
        r2 = r2_score(y_valid, valid_preds)
        
        rmse_scores.append(rmse)
        mae_scores.append(mae)
        r2_scores.append(r2)
        
        print(f"Fold {fold}")
        print("RMSE:", rmse)
        print("MAE :", mae)
        print("R2  :", r2)
        print("-" * 40)
    
    print("Average RMSE:", np.mean(rmse_scores))
    print("Average MAE :", np.mean(mae_scores))
    print("Average R2  :", np.mean(r2_scores))
    
    return {
        "rmse": np.mean(rmse_scores),
        "mae": np.mean(mae_scores),
        "r2": np.mean(r2_scores)
    }

Linear Regression Baseline

In [9]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_scores = cross_validate_model(linear_model, X, y)

Fold 1
RMSE: 0.24683494780746398
MAE : 0.19211864452760907
R2  : -0.06941765076021422
----------------------------------------
Fold 2
RMSE: 0.23949852056695775
MAE : 0.18569442662968477
R2  : -0.053456024009552294
----------------------------------------
Fold 3
RMSE: 0.24627571098605539
MAE : 0.18975458695462424
R2  : -0.0792310033570045
----------------------------------------
Fold 4
RMSE: 0.24242443737568029
MAE : 0.18943349048194422
R2  : -0.05796442401610302
----------------------------------------
Fold 5
RMSE: 0.24447925062623024
MAE : 0.18981757518125386
R2  : -0.09271056767454677
----------------------------------------
Average RMSE: 0.2439025734724775
Average MAE : 0.18936374475502324
Average R2  : -0.07055593396348417


This looks not suitable for submitting . Keep the file but no submit

In [10]:
linear_model.fit(X, y)

linear_preds = linear_model.predict(X_test)
linear_preds = np.clip(linear_preds, 0, 1)

sub_linear = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": linear_preds
})

validate_submission(sub_linear)

sub_linear.to_csv("../submissions/sub_v002_linear_regression.csv", index=False)

print("Saved: ../submissions/sub_v002_linear_regression.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.432091
1   F100765          0.528604
2   F107573          0.493944
3   F110345          0.480293
4   F118850          0.552281

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.1964644698660561
Max: 0.8031292677656026

Submission is valid.
Saved: ../submissions/sub_v002_linear_regression.csv


Ridge Regression Aphas select

In [11]:
ridge_results = []

alphas = [0.1, 1, 5, 10, 25, 50, 100, 200, 500, 1000]

for alpha in alphas:
    print("\n" + "=" * 60)
    print("Testing Ridge alpha:", alpha)
    
    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    
    scores = cross_validate_model(ridge_model, X, y)
    
    ridge_results.append({
        "alpha": alpha,
        "rmse": scores["rmse"],
        "mae": scores["mae"],
        "r2": scores["r2"]
    })

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df.sort_values("rmse")


Testing Ridge alpha: 0.1
Fold 1
RMSE: 0.24682538260553194
MAE : 0.19214147234392107
R2  : -0.06933476948532036
----------------------------------------
Fold 2
RMSE: 0.23950620301815628
MAE : 0.1857253156570307
R2  : -0.053523609014066054
----------------------------------------
Fold 3
RMSE: 0.24623763399933662
MAE : 0.18971156269543776
R2  : -0.07889730672382345
----------------------------------------
Fold 4
RMSE: 0.24229809647794917
MAE : 0.18933452226692535
R2  : -0.05686198280501564
----------------------------------------
Fold 5
RMSE: 0.2444924693376571
MAE : 0.18979979428780122
R2  : -0.09282873407207504
----------------------------------------
Average RMSE: 0.24387195708772622
Average MAE : 0.18934253345022323
Average R2  : -0.07028928042006011

Testing Ridge alpha: 1
Fold 1
RMSE: 0.2468106313931219
MAE : 0.19217319315338088
R2  : -0.06920695837611146
----------------------------------------
Fold 2
RMSE: 0.23951241132835654
MAE : 0.1857567381385642
R2  : -0.053578227107928145
-

,alpha,rmse,mae,r2
9,1000.0,0.242104,0.187775,-0.054824
8,500.0,0.242856,0.188475,-0.061390
7,200.0,0.243367,0.188937,-0.065862
6,100.0,0.243555,0.189101,-0.067508
5,50.0,0.243658,0.189189,-0.068418
4,25.0,0.243720,0.189241,-0.068961
3,10.0,0.243774,0.189284,-0.069429
2,5.0,0.243803,0.189305,-0.069690
1,1.0,0.243844,0.189329,-0.070049
0,0.1,0.243872,0.189343,-0.070289


In [12]:
ridge_results_extra = []

extra_alphas = [1500, 2000, 3000, 5000, 7500, 10000, 20000]

for alpha in extra_alphas:
    print("\n" + "=" * 60)
    print("Testing Ridge alpha:", alpha)
    
    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    
    scores = cross_validate_model(ridge_model, X, y)
    
    ridge_results_extra.append({
        "alpha": alpha,
        "rmse": scores["rmse"],
        "mae": scores["mae"],
        "r2": scores["r2"]
    })

ridge_results_extra_df = pd.DataFrame(ridge_results_extra)

all_ridge_results_df = pd.concat(
    [ridge_results_df, ridge_results_extra_df],
    axis=0
).sort_values("rmse")

all_ridge_results_df


Testing Ridge alpha: 1500
Fold 1
RMSE: 0.24442865910380637
MAE : 0.19005576759985163
R2  : -0.048668688765980406
----------------------------------------
Fold 2
RMSE: 0.23723580231470875
MAE : 0.1835615190910188
R2  : -0.03364451111668809
----------------------------------------
Fold 3
RMSE: 0.24353307401428215
MAE : 0.18741063589203577
R2  : -0.05532724740301087
----------------------------------------
Fold 4
RMSE: 0.2401588665860462
MAE : 0.1873700841157359
R2  : -0.038282470564805715
----------------------------------------
Fold 5
RMSE: 0.24185491129753525
MAE : 0.18735668519193913
R2  : -0.06937728202585158
----------------------------------------
Average RMSE: 0.2414422626632758
Average MAE : 0.18715093837811628
Average R2  : -0.04906003997526733

Testing Ridge alpha: 2000
Fold 1
RMSE: 0.24384845228264213
MAE : 0.18950273032386994
R2  : -0.043696091952973415
----------------------------------------
Fold 2
RMSE: 0.23668403738026247
MAE : 0.1829899165209488
R2  : -0.028841985083841

,alpha,rmse,mae,r2
6,20000.0,0.234525,0.180285,0.010218
5,10000.0,0.236144,0.181958,-0.003499
4,7500.0,0.237022,0.182847,-0.010979
3,5000.0,0.238317,0.184134,-0.022062
2,3000.0,0.239840,0.185612,-0.035180
1,2000.0,0.240851,0.186586,-0.043929
0,1500.0,0.241442,0.187151,-0.049060
9,1000.0,0.242104,0.187775,-0.054824
8,500.0,0.242856,0.188475,-0.061390
7,200.0,0.243367,0.188937,-0.065862


In [13]:
ridge_results_final = []

final_alphas = [30000, 50000, 75000, 100000, 150000]

for alpha in final_alphas:
    print("\n" + "=" * 60)
    print("Testing Ridge alpha:", alpha)
    
    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    
    scores = cross_validate_model(ridge_model, X, y)
    
    ridge_results_final.append({
        "alpha": alpha,
        "rmse": scores["rmse"],
        "mae": scores["mae"],
        "r2": scores["r2"]
    })

ridge_results_final_df = pd.DataFrame(ridge_results_final)

all_ridge_results_df = pd.concat(
    [all_ridge_results_df, ridge_results_final_df],
    axis=0
).sort_values("rmse")

all_ridge_results_df


Testing Ridge alpha: 30000
Fold 1
RMSE: 0.23702308470166114
MAE : 0.1824340950429828
R2  : 0.013912752954861052
----------------------------------------
Fold 2
RMSE: 0.2309657829306062
MAE : 0.17680564103621102
R2  : 0.020270848230833027
----------------------------------------
Fold 3
RMSE: 0.23555770719225175
MAE : 0.1804633456620809
R2  : 0.012661921325820669
----------------------------------------
Fold 4
RMSE: 0.23362955031819002
MAE : 0.1804635139364613
R2  : 0.01740665719868828
----------------------------------------
Fold 5
RMSE: 0.23302859641013005
MAE : 0.17870645810574803
R2  : 0.007250743673086202
----------------------------------------
Average RMSE: 0.23404094431056782
Average MAE : 0.17977461075669682
Average R2  : 0.014300584676657846

Testing Ridge alpha: 50000
Fold 1
RMSE: 0.23687016096218427
MAE : 0.18226534056976765
R2  : 0.015184759888289934
----------------------------------------
Fold 2
RMSE: 0.23103781202032264
MAE : 0.17687749184213014
R2  : 0.01965967542493596

,alpha,rmse,mae,r2
1,50000.0,0.233897,0.179644,0.015511
0,30000.0,0.234041,0.179775,0.014301
2,75000.0,0.234045,0.179824,0.014265
3,100000.0,0.234229,0.180034,0.012721
6,20000.0,0.234525,0.180285,0.010218
4,150000.0,0.234529,0.180379,0.010192
5,10000.0,0.236144,0.181958,-0.003499
4,7500.0,0.237022,0.182847,-0.010979
3,5000.0,0.238317,0.184134,-0.022062
2,3000.0,0.239840,0.185612,-0.035180


In [14]:
best_alpha = 50000

print("Selected best Ridge alpha:", best_alpha)

Selected best Ridge alpha: 50000


In [15]:
best_ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_alpha))
])

best_ridge_model.fit(X, y)

print("Final Ridge model trained using all training data.")

Final Ridge model trained using all training data.


In [16]:
ridge_preds = best_ridge_model.predict(X_test)

print("Before clipping:")
print("Minimum prediction:", ridge_preds.min())
print("Maximum prediction:", ridge_preds.max())

Before clipping:
Minimum prediction: 0.3588329756483521
Maximum prediction: 0.59584216574293


In [17]:
ridge_preds = np.clip(ridge_preds, 0, 1)

print("After clipping:")
print("Minimum prediction:", ridge_preds.min())
print("Maximum prediction:", ridge_preds.max())

After clipping:
Minimum prediction: 0.3588329756483521
Maximum prediction: 0.59584216574293


In [18]:
sub_ridge = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": ridge_preds
})

validate_submission(sub_ridge)

sub_ridge.to_csv("../submissions/sub_v003_ridge_alpha50000.csv", index=False)

print("Saved: ../submissions/sub_v003_ridge_alpha50000.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.473541
1   F100765          0.495558
2   F107573          0.488214
3   F110345          0.481583
4   F118850          0.502954

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.3588329756483521
Max: 0.59584216574293

Submission is valid.
Saved: ../submissions/sub_v003_ridge_alpha50000.csv


In [19]:
train_mean = y.mean()

ridge_preds_conservative_090 = 0.90 * ridge_preds + 0.10 * train_mean
ridge_preds_conservative_090 = np.clip(ridge_preds_conservative_090, 0, 1)

sub_ridge_conservative_090 = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": ridge_preds_conservative_090
})

validate_submission(sub_ridge_conservative_090)

sub_ridge_conservative_090.to_csv(
    "../submissions/sub_v004_ridge_alpha50000_conservative_090.csv",
    index=False
)

print("Saved: ../submissions/sub_v004_ridge_alpha50000_conservative_090.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.474038
1   F100765          0.493854
2   F107573          0.487244
3   F110345          0.481276
4   F118850          0.500510

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.3708014204693037
Max: 0.5841096915544238

Submission is valid.
Saved: ../submissions/sub_v004_ridge_alpha50000_conservative_090.csv


In [20]:
ridge_preds_conservative_095 = 0.95 * ridge_preds + 0.05 * train_mean
ridge_preds_conservative_095 = np.clip(ridge_preds_conservative_095, 0, 1)

sub_ridge_conservative_095 = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": ridge_preds_conservative_095
})

validate_submission(sub_ridge_conservative_095)

sub_ridge_conservative_095.to_csv(
    "../submissions/sub_v005_ridge_alpha50000_conservative_095.csv",
    index=False
)

print("Saved: ../submissions/sub_v005_ridge_alpha50000_conservative_095.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.473789
1   F100765          0.494706
2   F107573          0.487729
3   F110345          0.481429
4   F118850          0.501732

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.3648171980588279
Max: 0.5899759286486769

Submission is valid.
Saved: ../submissions/sub_v005_ridge_alpha50000_conservative_095.csv


In [21]:
prediction_comparison = pd.DataFrame({
    "ridge_normal": ridge_preds,
    "ridge_conservative_095": ridge_preds_conservative_095,
    "ridge_conservative_090": ridge_preds_conservative_090
})

prediction_comparison.describe()

,ridge_normal,ridge_conservative_095,ridge_conservative_090
count,5300.000000,5300.000000,5300.000000
mean,0.482139,0.481958,0.481777
std,0.027841,0.026449,0.025057
min,0.358833,0.364817,0.370801
25%,0.463741,0.464480,0.465219
50%,0.482196,0.482012,0.481828
75%,0.500990,0.499867,0.498743
max,0.595842,0.589976,0.584110


In [22]:
print("Normal Ridge:")
print("0 predictions:", (ridge_preds == 0).sum())
print("1 predictions:", (ridge_preds == 1).sum())

print("\nConservative 095:")
print("0 predictions:", (ridge_preds_conservative_095 == 0).sum())
print("1 predictions:", (ridge_preds_conservative_095 == 1).sum())

print("\nConservative 090:")
print("0 predictions:", (ridge_preds_conservative_090 == 0).sum())
print("1 predictions:", (ridge_preds_conservative_090 == 1).sum())

Normal Ridge:
0 predictions: 0
1 predictions: 0

Conservative 095:
0 predictions: 0
1 predictions: 0

Conservative 090:
0 predictions: 0
1 predictions: 0


In [24]:
ridge_score_summary = pd.DataFrame([
    {
        "version": "sub_v003_ridge_alpha50000",
        "model": "Ridge Regression",
        "alpha": best_alpha,
        "cv_rmse": 0.233897,
        "cv_mae": 0.179644,
        "cv_r2": 0.015511,
        "file": "../submissions/sub_v003_ridge_alpha50000.csv",
        "notes": "Best Ridge alpha from CV"
    },
    {
        "version": "sub_v004_ridge_alpha50000_conservative_090",
        "model": "Ridge Regression + shrinkage",
        "alpha": best_alpha,
        "cv_rmse": None,
        "cv_mae": None,
        "cv_r2": None,
        "file": "../submissions/sub_v004_ridge_alpha50000_conservative_090.csv",
        "notes": "90% Ridge + 10% train mean"
    },
    {
        "version": "sub_v005_ridge_alpha50000_conservative_095",
        "model": "Ridge Regression + shrinkage",
        "alpha": best_alpha,
        "cv_rmse": None,
        "cv_mae": None,
        "cv_r2": None,
        "file": "../submissions/sub_v005_ridge_alpha50000_conservative_095.csv",
        "notes": "95% Ridge + 5% train mean"
    }
])

ridge_score_summary.to_csv("../reports/ridge_submission_summary.csv", index=False)

ridge_score_summary

,version,model,alpha,cv_rmse,cv_mae,cv_r2,file,notes
0,sub_v003_ridge_alpha50000,Ridge Regression,50000,0.233897,0.179644,0.015511,../submissions/sub_v003_ridge_alpha50000.csv,Best Ridge alpha from CV
1,sub_v004_ridge_alpha50000_conservative_090,Ridge Regression + shrinkage,50000,NaN,NaN,NaN,../submissions/sub_v004_ridge_alpha50000_conse...,90% Ridge + 10% train mean
2,sub_v005_ridge_alpha50000_conservative_095,Ridge Regression + shrinkage,50000,NaN,NaN,NaN,../submissions/sub_v005_ridge_alpha50000_conse...,95% Ridge + 5% train mean
